# Gold — Métricas de negocio
**Proyecto:** AI Tech Landscape Pipeline  
**Descripción:** Calcula las métricas finales de negocio listas para visualización en Power BI.

In [1]:
import pandas as pd
import os

SILVER_PATH = "../data/silver/"
GOLD_PATH   = "../data/gold/"
os.makedirs(GOLD_PATH, exist_ok=True)

df_stocks = pd.read_parquet(f"{SILVER_PATH}silver_big_tech_stocks.parquet")
df_gpu    = pd.read_parquet(f"{SILVER_PATH}silver_gpu_companies.parquet")
df_ai     = pd.read_parquet(f"{SILVER_PATH}silver_ai_companies.parquet")

print("✓ Datasets cargados")

✓ Datasets cargados


## Big Tech — Precio promedio anual y variación %
Responde: ¿Cómo creció cada empresa año a año?

In [2]:
df_stocks["year"] = df_stocks["date"].dt.year

# Precio promedio de cierre por empresa por año
gold_precio_anual = (
    df_stocks.groupby(["ticker", "year"])["close"]
    .mean()
    .reset_index()
    .rename(columns={"close": "avg_close_price"})
)

# Variación % vs año anterior
gold_precio_anual["pct_change_yoy"] = (
    gold_precio_anual
    .groupby("ticker")["avg_close_price"]
    .pct_change() * 100
).round(2)

gold_precio_anual["avg_close_price"] = gold_precio_anual["avg_close_price"].round(2)

print(f"Filas: {len(gold_precio_anual):,}")
gold_precio_anual.head(10)

Filas: 181


,ticker,year,avg_close_price,pct_change_yoy
0,AAPL,2010,9.28,NaN
1,AAPL,2011,13.00,40.09
2,AAPL,2012,20.57,58.25
3,AAPL,2013,16.88,-17.95
4,AAPL,2014,23.07,36.65
5,AAPL,2015,30.01,30.10
6,AAPL,2016,26.15,-12.86
7,AAPL,2017,37.64,43.92
8,AAPL,2018,47.26,25.57
9,AAPL,2019,52.06,10.16


## Big Tech — Mejor y peor año por empresa
Responde: ¿Cuándo tuvo cada empresa su pico y su caída más grande?

In [3]:
mejor_anio = (
    gold_precio_anual.loc[gold_precio_anual.groupby("ticker")["pct_change_yoy"].idxmax()]
    [["ticker", "year", "pct_change_yoy"]]
    .rename(columns={"year": "mejor_anio", "pct_change_yoy": "mejor_retorno_pct"})
)

peor_anio = (
    gold_precio_anual.loc[gold_precio_anual.groupby("ticker")["pct_change_yoy"].idxmin()]
    [["ticker", "year", "pct_change_yoy"]]
    .rename(columns={"year": "peor_anio", "pct_change_yoy": "peor_retorno_pct"})
)

gold_performance = mejor_anio.merge(peor_anio, on="ticker")
print(f"Filas: {len(gold_performance)}")
gold_performance

Filas: 14


,ticker,mejor_anio,mejor_retorno_pct,peor_anio,peor_retorno_pct
0,AAPL,2020,83.13,2013,-17.95
1,ADBE,2018,63.22,2022,-29.30
2,AMZN,2018,69.57,2022,-24.48
3,CRM,2018,46.48,2022,-28.51
4,CSCO,2018,33.43,2011,-25.17
5,GOOGL,2021,67.99,2022,-7.53
6,IBM,2011,29.62,2015,-14.78
7,INTC,2014,31.52,2022,-30.81
8,META,2014,93.80,2022,-43.90
9,MSFT,2020,48.05,2011,-3.72


## GPU Race — NVIDIA vs AMD vs INTEL evolución histórica
Responde: ¿Quién gana la carrera del hardware de IA?

In [4]:
df_gpu["year"] = df_gpu["date"].dt.year

gold_gpu = (
    df_gpu.groupby(["empresa", "year"])["close"]
    .mean()
    .reset_index()
    .rename(columns={"close": "avg_close_price"})
)

gold_gpu["pct_change_yoy"] = (
    gold_gpu
    .groupby("empresa")["avg_close_price"]
    .pct_change() * 100
).round(2)

gold_gpu["avg_close_price"] = gold_gpu["avg_close_price"].round(2)

print(f"Empresas: {sorted(gold_gpu['empresa'].unique())}")
print(f"Filas   : {len(gold_gpu):,}")
gold_gpu.tail(10)

Empresas: ['AMD', 'ASUS', 'INTEL', 'MSI', 'NVIDIA']
Filas   : 143


,empresa,year,avg_close_price,pct_change_yoy
133,NVIDIA,2015,5.93,27.88
134,NVIDIA,2016,13.44,126.67
135,NVIDIA,2017,37.45,178.61
136,NVIDIA,2018,58.10,55.15
137,NVIDIA,2019,43.65,-24.87
138,NVIDIA,2020,98.91,126.61
139,NVIDIA,2021,195.22,97.38
140,NVIDIA,2022,185.69,-4.88
141,NVIDIA,2023,336.52,81.23
142,NVIDIA,2024,736.97,119.00


## AI Companies — Top 10 por revenue
Responde: ¿Quién factura más en el sector de IA?

In [5]:
gold_top_revenue = (
    df_ai[["company_name", "revenue_usd", "founded", "headquarters"]]
    .dropna(subset=["revenue_usd"])
    .sort_values("revenue_usd", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

gold_top_revenue["revenue_usd_billions"] = (gold_top_revenue["revenue_usd"] / 1e9).round(2)

print(f"Top 10 empresas AI por revenue:")
gold_top_revenue[["company_name", "revenue_usd_billions", "founded", "headquarters"]]

Top 10 empresas AI por revenue:


,company_name,revenue_usd_billions,founded,headquarters
0,Amazon,574.79,1994,"Seattle, Washington; Arlington, Virginia"
1,Google,305.60,1998,"Mountain View, California"
2,Microsoft,211.00,1975,"Redmond, Washington"
3,Tesla,96.77,2003,"Austin, Texas"
4,Tencent,85.00,1998,"Shenzhen, China"
5,Siemens,83.65,1847,"Munich, Germany"
6,GE Vernova,68.00,2022,"Cambridge, Massachusets"
7,IBM (International Business Machines Corporation),61.90,1911,"Armonk, New York"
8,Oracle,49.95,1977,"Austin, Texas"
9,SAP,33.54,1972,"Walldorf, Germany"


## AI Companies — Revenue vs Glassdoor Score
Responde: ¿Las empresas que más facturan son las mejores para trabajar?

In [6]:
gold_culture = (
    df_ai[["company_name", "revenue_usd", "glassdoor_score", "founded", "headquarters"]]
    .dropna(subset=["revenue_usd", "glassdoor_score"])
    .copy()
)

gold_culture["revenue_usd_billions"] = (gold_culture["revenue_usd"] / 1e9).round(2)
gold_culture["company_age"] = 2024 - gold_culture["founded"].astype(float)

print(f"Empresas con revenue y glassdoor: {len(gold_culture)}")
gold_culture[["company_name", "revenue_usd_billions", "glassdoor_score", "company_age"]].head(10)

Empresas con revenue y glassdoor: 92


,company_name,revenue_usd_billions,glassdoor_score,company_age
0,Alibaba Cloud,0.48,3.7,15.0
1,DataRobot,0.34,3.7,12.0
2,Google,305.60,4.4,26.0
3,Hugging Face,0.04,4.3,8.0
4,H2O.ai,0.07,3.1,13.0
5,Rasa,0.02,3.9,8.0
7,ActiveCampaign,0.20,3.7,21.0
8,ClickUp,0.16,3.4,7.0
9,Freshworks,0.60,3.8,14.0
10,HubSpot,2.17,4.1,18.0


## Guardar Gold

In [7]:
gold_precio_anual.to_parquet(f"{GOLD_PATH}gold_bigtech_precio_anual.parquet",  index=False)
gold_performance.to_parquet(f"{GOLD_PATH}gold_bigtech_performance.parquet",    index=False)
gold_gpu.to_parquet(f"{GOLD_PATH}gold_gpu_race.parquet",                       index=False)
gold_top_revenue.to_parquet(f"{GOLD_PATH}gold_ai_top_revenue.parquet",         index=False)
gold_culture.to_parquet(f"{GOLD_PATH}gold_ai_culture_vs_money.parquet",        index=False)

print("=" * 50)
print("GOLD COMPLETADO")
print("=" * 50)
print(f"  Big Tech precio anual : {len(gold_precio_anual):,} filas")
print(f"  Big Tech performance  : {len(gold_performance):,} filas")
print(f"  GPU Race              : {len(gold_gpu):,} filas")
print(f"  AI Top Revenue        : {len(gold_top_revenue):,} filas")
print(f"  AI Culture vs Money   : {len(gold_culture):,} filas")
print("\n✓ Parquet listos para Power BI →")

GOLD COMPLETADO
  Big Tech precio anual : 181 filas
  Big Tech performance  : 14 filas
  GPU Race              : 143 filas
  AI Top Revenue        : 10 filas
  AI Culture vs Money   : 92 filas

✓ Parquet listos para Power BI →
